### SETUP - Dataset ISIC 2019

Generamos dos subconjuntos de datos a partir del dataset ISIC 2019:
- **Entrenamiento**: 100 imagenes de cada clase
- **Test**: 10 imagenes de cada clase

**IMPORTANTE**: Si el dataset ya existe en `dataset_isic/`, no es necesario volver a descargar nada. 
El script detecta automaticamente si las imagenes ya estan disponibles.

In [ ]:
# Importaciones
import os
import shutil
import pandas as pd
import zipfile
import requests
from tqdm import tqdm  # Para barra de progreso en descarga

### Configuración

In [ ]:
# Configuracion de rutas y URLs
NOMBRE_ZIP = "ISIC_2019_Training_Input.zip" 

# URLs oficiales del dataset ISIC 2019
URL_DATA = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip"
URL_LABELS = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv"

# Directorios de destino
BASE_DIR = "dataset_isic"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
CSV_PATH = "ground_truth.csv"

# Clases esperadas del dataset ISIC 2019
CLASES_ESPERADAS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
N_TRAIN_POR_CLASE = 100
N_TEST_POR_CLASE = 10

def verificar_dataset_completo():
    # Verifica si el dataset ya esta completo
    for clase in CLASES_ESPERADAS:
        train_path = os.path.join(TRAIN_DIR, clase)
        test_path = os.path.join(TEST_DIR, clase)
        
        if not os.path.exists(train_path) or not os.path.exists(test_path):
            return False
        
        n_train = len([f for f in os.listdir(train_path) if f.endswith('.jpg')])
        n_test = len([f for f in os.listdir(test_path) if f.endswith('.jpg')])
        
        # Verificamos que haya al menos las imagenes esperadas (o todas si hay menos)
        if n_train < min(N_TRAIN_POR_CLASE, 1) or n_test < min(N_TEST_POR_CLASE, 1):
            return False
    
    return True

# Verificar estado del dataset
DATASET_COMPLETO = verificar_dataset_completo()

if DATASET_COMPLETO:
    print("Dataset ya existe y esta completo.")
    print("No es necesario descargar nada.")
    print(f"\nUbicacion: {os.path.abspath(BASE_DIR)}")
else:
    print("Dataset incompleto o no existe. Se procedera a descargar.")
    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(TEST_DIR, exist_ok=True)
    print(f"  - Train: {TRAIN_DIR}")
    print(f"  - Test: {TEST_DIR}")

### Descarga de Datos (solo si es necesario)

In [ ]:
if DATASET_COMPLETO:
    print("Saltando descarga - dataset ya completo.")
else:
    def descargar_archivo(url, destino, descripcion="Descargando"):
        """Descarga un archivo desde URL con barra de progreso."""
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        
        with open(destino, 'wb') as f:
            if total_size == 0:
                f.write(response.content)
            else:
                with tqdm(total=total_size, unit='B', unit_scale=True, desc=descripcion) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

    # Descargar CSV si no existe
    if not os.path.exists(CSV_PATH):
        print("Descargando etiquetas (CSV)")
        descargar_archivo(URL_LABELS, CSV_PATH, "CSV Etiquetas")
        print("CSV descargado correctamente.")
    else:
        print(f"CSV ya existe: {CSV_PATH}")

    # Descargar ZIP si no existe
    if not os.path.exists(NOMBRE_ZIP):
        print(f"\nDescargando imagenes (ZIP de aproximadamente 9GB)")
        print("Esto puede tardar varios minutos dependiendo de tu conexion")
        descargar_archivo(URL_DATA, NOMBRE_ZIP, "ZIP Imagenes")
        print("ZIP descargado correctamente.")
    else:
        print(f"ZIP ya existe: {NOMBRE_ZIP}")

    print("\nArchivos de origen listos.")

### Extracción Selectiva y Creación de Conjuntos

Extraemos únicamente las imágenes necesarias del ZIP:
- **100 imágenes por clase** para entrenamiento
- **10 imágenes por clase** para test

In [ ]:
if DATASET_COMPLETO:
    print("Saltando extraccion - dataset ya completo.")
else:
    # Cargar etiquetas
    df = pd.read_csv(CSV_PATH)

    # Obtener clases (todas las columnas menos 'image' y 'UNK')
    clases = [c for c in df.columns if c not in ['image', 'UNK']]
    print(f"Clases detectadas: {clases}")

    print("\nIniciando extraccion selectiva de subconjuntos:")

    with zipfile.ZipFile(NOMBRE_ZIP, 'r') as zip_ref:
        # Crear mapa de archivos dentro del ZIP para busqueda rapida
        print("Mapeando contenido del ZIP (unos segundos)")
        all_files_map = {os.path.basename(f): f for f in zip_ref.namelist()}
        
        for clase in clases:
            print(f"\nProcesando clase: {clase}")
            
            # Crear carpetas destino por clase
            os.makedirs(os.path.join(TRAIN_DIR, clase), exist_ok=True)
            os.makedirs(os.path.join(TEST_DIR, clase), exist_ok=True)

            # Filtrar imagenes de la clase actual
            imgs_clase = df[df[clase] == 1.0]['image'].tolist()
            
            # Seleccion aleatoria de 110 imagenes (100 train + 10 test)
            if len(imgs_clase) < 110:
                print(f"  Aviso: {clase} tiene solo {len(imgs_clase)} imagenes. Se usaran todas.")
                seleccion = imgs_clase
            else:
                seleccion = df[df[clase] == 1.0]['image'].sample(n=110, random_state=42).tolist()
            
            lista_train = seleccion[:100]
            lista_test = seleccion[100:110]

            # Funcion de extraccion
            def extraer_lista(lista, carpeta_destino):
                count = 0
                for img_name in lista:
                    filename = img_name + ".jpg"
                    ruta_en_zip = all_files_map.get(filename)
                    
                    if ruta_en_zip:
                        source = zip_ref.open(ruta_en_zip)
                        target = open(os.path.join(carpeta_destino, filename), "wb")
                        with source, target:
                            shutil.copyfileobj(source, target)
                        count += 1
                return count

            n_train = extraer_lista(lista_train, os.path.join(TRAIN_DIR, clase))
            n_test = extraer_lista(lista_test, os.path.join(TEST_DIR, clase))
            
            print(f"  -> Train: {n_train} | Test: {n_test}")

    print("\nExtraccion completada.")